# Chapter 9: Building Complex Prompts from Scratch

## Lesson

Now we combine everything! A well-structured complex prompt can include up to **10 elements** (though most prompts don't need all of them):

1. **Task context** — What is the broader goal?
2. **Tone context** — How should Claude communicate?
3. **Detailed task description** — Specific rules and requirements
4. **Examples** — Ideal input/output pairs
5. **Input data** — The actual data to process (in XML tags)
6. **Immediate task description** — What to do right now
7. **Precognition instructions** — Think step-by-step
8. **Output formatting** — How to structure the response
9. **Prefill** — Start Claude's response

Not every prompt needs every element. Use what makes sense for your task.

In [ ]:
import anthropic

%store -r API_KEY
%store -r MODEL_NAME

client = anthropic.Anthropic(api_key=API_KEY)

def get_completion(prompt: str, system_prompt: str = "", prefill: str = ""):
    messages = [{"role": "user", "content": prompt}]
    if prefill:
        messages.append({"role": "assistant", "content": prefill})
    kwargs = {
        "model": MODEL_NAME,
        "max_tokens": 4000,
        "temperature": 0.0,
        "messages": messages
    }
    if system_prompt:
        kwargs["system"] = system_prompt
    message = client.messages.create(**kwargs)
    return message.content[0].text

### Example: Career Coach Chatbot

Here's a complex prompt that combines many of the techniques we've learned:

In [ ]:
SYSTEM_PROMPT = """You are an experienced career coach with 20 years of experience 
helping professionals navigate career transitions. You're warm, encouraging, 
but also honest and practical."""

USER_MESSAGE = "I'm a software engineer with 5 years of experience thinking about moving into product management."

PROMPT = f"""You are helping professionals make career decisions.

TONE: Be supportive and encouraging, but realistic. Avoid sugarcoating challenges.

RULES:
- Always ask at least one clarifying question
- Mention both pros and cons of any career move
- Reference specific, actionable steps
- Keep responses under 300 words

<example>
User: I want to switch from marketing to data science.
Coach: That's an exciting direction! Marketing actually gives you a great foundation — 
you already understand business metrics and customer behavior, which are huge in data science.

Here's what I'd consider:
**Strengths you bring:** Business acumen, storytelling with data, stakeholder management
**Gaps to fill:** Programming (Python/R), statistics, machine learning fundamentals

A realistic path: Start with an online Python course and a stats refresher (3-6 months), 
then build 2-3 portfolio projects using marketing data you're familiar with.

Quick question: Are you drawn more to the analysis side or the engineering/ML side?
</example>

<user_message>{USER_MESSAGE}</user_message>

Think about the user's background and the target role before responding.
Structure your response with bolded headers for key sections."""

response = get_completion(PROMPT, system_prompt=SYSTEM_PROMPT)
print(response)

### Example: Legal Document Analyzer

A more structured prompt for professional document analysis:

In [ ]:
CONTRACT_EXCERPT = """Section 4.2 - Termination for Convenience
Either party may terminate this Agreement upon thirty (30) days' written notice 
to the other party. Upon such termination, Client shall pay for all services 
rendered up to the effective date of termination.

Section 4.3 - Termination for Cause
Either party may terminate this Agreement immediately upon written notice if 
the other party materially breaches any term of this Agreement and fails to 
cure such breach within fifteen (15) days after receiving written notice thereof.

Section 7.1 - Limitation of Liability
In no event shall either party be liable for any indirect, incidental, special, 
or consequential damages. The total liability of either party shall not exceed 
the fees paid in the twelve (12) months preceding the claim."""

QUESTION = "If we want to end this contract, what are our options and timelines?"

PROMPT = f"""You are a legal document analyst helping a business client understand contracts.

RULES:
- Always cite specific section numbers using [Section X.X] format
- Use plain language — avoid legal jargon when possible
- Flag any potential risks or important caveats
- Never provide legal advice — frame as analysis only

<contract>{CONTRACT_EXCERPT}</contract>

<question>{QUESTION}</question>

First, extract relevant quotes in <quotes> tags.
Then provide your analysis in <analysis> tags."""

response = get_completion(PROMPT)
print(response)

---
## Exercises

### Exercise 9.1: Financial Services Chatbot
Build a financial analysis assistant that helps users understand tax documents. Fill in the template below:

In [ ]:
# Exercise 9.1
TAX_INFO = """Standard Deduction Amounts (2024):
- Single: $14,600
- Married Filing Jointly: $29,200
- Head of Household: $21,900

Tax Brackets (Single Filers):
- 10%: $0 - $11,600
- 12%: $11,601 - $47,150
- 22%: $47,151 - $100,525
- 24%: $100,526 - $191,950
- 32%: $191,951 - $243,725
- 35%: $243,726 - $609,350
- 37%: Over $609,350"""

USER_QUESTION = "I'm single and made $85,000 this year. How much will I owe in federal taxes approximately?"

# TODO: Build a comprehensive prompt using all the techniques from this course
SYSTEM_PROMPT = ""  # Task context + tone
PROMPT = ""  # Detailed instructions + data + question + output formatting
PREFILL = ""  # Optional: guide the response

response = get_completion(PROMPT, system_prompt=SYSTEM_PROMPT, prefill=PREFILL)
print(response)

# Grading
def grade_exercise_9_1(response):
    resp_lower = response.lower()
    has_numbers = any(c.isdigit() for c in response)
    has_structure = "<" in response or "**" in response or "#" in response
    has_disclaimer = any(w in resp_lower for w in ["not", "consult", "professional", "estimate", "approximate"])
    return has_numbers and (has_structure or has_disclaimer)

print("\n" + ("✅ PASS" if grade_exercise_9_1(response) else "❌ TRY AGAIN"))

### Exercise 9.2: Code Review Tutor (Socratic Method)
Build a code review assistant that uses the **Socratic method** — guiding users to find issues themselves rather than giving direct answers.

In [ ]:
# Exercise 9.2
CODE_SNIPPET = """
def calculate_average(numbers):
    total = 0
    for n in numbers:
        total += n
    average = total / len(numbers)
    return average

result = calculate_average([])
print(f"Average: {result}")
"""

# TODO: Build a Socratic code review prompt
# It should:
# - Identify potential issues in <issues> tags
# - Guide the user with questions rather than direct answers in <response> tags
# - Be encouraging and educational

SYSTEM_PROMPT = ""  # Your system prompt
PROMPT = ""  # Your prompt

response = get_completion(PROMPT, system_prompt=SYSTEM_PROMPT)
print(response)

# Grading
def grade_exercise_9_2(response):
    resp_lower = response.lower()
    has_question = "?" in response
    mentions_issue = "empty" in resp_lower or "zero" in resp_lower or "len" in resp_lower
    return has_question and mentions_issue

print("\n" + ("✅ PASS" if grade_exercise_9_2(response) else "❌ TRY AGAIN — Should ask questions and identify the empty list issue"))

---
## Congratulations!

You've completed the Prompt Engineering Tutorial! Here's a summary of what you've learned:

| Chapter | Technique | Key Takeaway |
|---------|-----------|-------------|
| 1 | Basic Structure | Messages API, system prompts, role alternation |
| 2 | Clarity | Be specific, eliminate ambiguity, direct instructions |
| 3 | Role Prompting | Assign roles via system prompt for better performance |
| 4 | Data Separation | Use XML tags to separate data from instructions |
| 5 | Output Formatting | Prefilling, structured output, XML/JSON responses |
| 6 | Precognition | Think step-by-step before answering |
| 7 | Few-Shot | Provide examples to control format and tone |
| 8 | Hallucinations | Give Claude an out, require citations |
| 9 | Complex Prompts | Combine all techniques into comprehensive prompts |

### Next Steps
- Practice building prompts for your own use cases
- Explore prompt chaining (breaking complex tasks into multiple calls)
- Learn about tool use (letting Claude call external functions)
- Check out the [Anthropic Documentation](https://docs.anthropic.com) for more

---
### Example Playground

In [ ]:
# Final Playground - build your own complex prompt!
SYSTEM_PROMPT = ""
PROMPT = ""
PREFILL = ""

if PROMPT:
    response = get_completion(PROMPT, system_prompt=SYSTEM_PROMPT, prefill=PREFILL)
    print(response)
else:
    print("Write your own complex prompt above and run this cell!")